# NB 6.1 &mdash; La lupa

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT6** &mdash; Xarxes neuronals convolucionals i introducció a la visió per computador

---
### D'on venim

A la secció 10 del NB 1.2 vam moure un dígit tres píxels i l'MLP va deixar de
reconèixer-lo: cap model no arribava al 55% d'encert. Per a l'MLP, una imatge és una
fila de 784 números, i no sap que dos píxels són veïns.

### Què farem avui

Provarem la xarxa que hi vam prometre: la **xarxa convolucional**, o **CNN**. Primer
l'entrenarem sense saber encara com funciona, i després l'obrirem per dins. No la
programarem: la configurarem amb paràmetres, com fèiem amb `MLPClassifier`, i la
mesurarem. **Bona part de la feina és teva:** hi ha sis missions pel camí.

### Preguntes que has de saber respondre en acabar

1. Què fa un filtre convolucional, i per què una CNN necessita tan pocs pesos per mirar tota la imatge?
2. Què fa el pooling, i què passa si el treus?
3. És cert que a una CNN no li importa on és el dígit? De què depèn?
4. Què passa amb l'avantatge de la CNN si barregem els píxels?

## 0. Preparació

Aquí apareix una biblioteca nova: **PyTorch**, una de les més usades per entrenar xarxes
neuronals. A scikit-learn, `MLPClassifier` amagava l'entrenament dins de `fit`. Aquí el
tenim a la vista en una cel·la que **no cal modificar**: només cal saber què fa cada funció.

In [ ]:
import io
import time
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import TensorDataset

URL_DATA = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT01-Xarxes_neuronals_supervisades/mnist/mnist.npz"

plt.rcParams["figure.figsize"] = (6, 4)

La cel·la següent és la **plantilla d'entrenament**. Hi ha `make_cnn`, que crea la xarxa,
`train_model`, que l'entrena, i `accuracy`, que en mesura l'encert. Dins de `train_model`
reconeixeràs les quatre passes de la UT5: predir, mesurar la loss, repartir la culpa
amb la backpropagation i fer una passa de gradient descent. Executa-la sense patir per
les línies que no coneguis.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Plantilla d'entrenament. No cal modificar-la: només cal saber què fa cada funció.
# Fa el mateix que MLPClassifier feia per dins a la UT5, però a la vista.
# ─────────────────────────────────────────────────────────────────────────────
import math
import time

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # GPU si n'hi ha; si no, CPU


def make_cnn(in_channels, n_classes, image_size=28, filters=(16, 32), pooling=True,
             global_pool=False, batch_norm=False, dropout=0.0):
    """Crea una CNN a partir de paràmetres, com MLPClassifier amb hidden_layer_sizes.

    Paràmetres:
        in_channels: canals de la imatge (1 en gris, 3 en color).
        n_classes: quantes classes ha de distingir.
        image_size: costat de la imatge en píxels.
        filters: filtres de cada capa convolucional; (16, 32) són dues capes.
        pooling: si cada capa redueix la imatge a la meitat amb MaxPool2d.
        global_pool: si al final es fa la mitjana de cada mapa en lloc d'aplanar-los.
        batch_norm: si s'afegeix BatchNorm2d després de cada convolució.
        dropout: fracció de neurones que s'apaguen a l'atzar a l'entrenament.

    Retorna el model. Exemple: model = make_cnn(1, 10, filters=(16, 32, 64))
    """
    n_pools = (len(filters) - 1 if global_pool else len(filters)) if pooling else 0
    side = image_size // 2 ** n_pools
    if side < 1:
        raise ValueError(f"Hi ha massa capes amb pooling per a una imatge de {image_size} píxels: "
                         "fes filters més curt o prova pooling=False.")

    torch.manual_seed(SEED)
    layers = []
    channels = in_channels
    for position, n_filters in enumerate(filters):
        layers += [nn.Conv2d(channels, n_filters, kernel_size=3, padding=1)]
        if batch_norm:
            layers += [nn.BatchNorm2d(n_filters)]
        layers += [nn.ReLU()]
        is_last = position == len(filters) - 1
        if pooling and not (global_pool and is_last):
            layers += [nn.MaxPool2d(2)]
        channels = n_filters

    switch_off = [nn.Dropout(dropout)] if dropout > 0 else []
    if global_pool:
        layers += [nn.AdaptiveAvgPool2d(1), nn.Flatten(), *switch_off, nn.Linear(channels, n_classes)]
    else:
        layers += [nn.Flatten(), nn.Linear(channels * side * side, 64), nn.ReLU(),
                   *switch_off, nn.Linear(64, n_classes)]
    return nn.Sequential(*layers)


def count_parameters(model):
    """Retorna quants pesos entrenables té el model."""
    return sum(weights.numel() for weights in model.parameters() if weights.requires_grad)


def make_loaders(train_set, val_set, batch_size=64, augment=None):
    """Prepara els mini-batches d'entrenament i de validació.

    Paràmetres:
        train_set, val_set: conjunts de dades de PyTorch.
        batch_size: imatges per mini-batch.
        augment: transformació per a cada imatge d'entrenament (data augmentation) o None.

    Retorna (train_loader, val_loader). Només l'entrenament es barreja i s'augmenta.
    """
    def collate_augmented(batch):
        images = torch.stack([augment(image) for image, _ in batch])
        labels = torch.as_tensor([label for _, label in batch])
        return images, labels

    collate = collate_augmented if augment is not None else None
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_set, batch_size=1000)
    return train_loader, val_loader


@torch.no_grad()
def _evaluate(model, loader):
    """Retorna la loss mitjana i l'encert del model amb un loader."""
    model.eval()
    device = next(model.parameters()).device
    loss_function = nn.CrossEntropyLoss(reduction="sum")
    total_loss, hits, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        scores = model(images)
        total_loss += loss_function(scores, labels).item()
        hits += (scores.argmax(dim=1) == labels).sum().item()
        total += len(labels)
    return total_loss / total, hits / total


def accuracy(model, loader):
    """Retorna l'encert del model (de 0 a 1), com score() a scikit-learn."""
    return _evaluate(model, loader)[1]


def train_model(model, train_loader, val_loader, epochs, lr, optimizer="adam"):
    """Entrena el model i mostra com va, epoch a epoch. És el fit() d'aquesta unitat.

    Paràmetres:
        model: la xarxa, feta amb make_cnn.
        train_loader, val_loader: els mini-batches, fets amb make_loaders.
        epochs: quantes vegades es recorren totes les dades d'entrenament.
        lr: el learning rate.
        optimizer: "adam" o "sgd".

    Retorna un diccionari amb les llistes train_loss, val_loss, train_acc i val_acc.
    """
    optimizers = {"adam": torch.optim.Adam, "sgd": torch.optim.SGD}
    if optimizer not in optimizers:
        raise ValueError(f"Optimitzador desconegut: {optimizer!r}. Tria 'adam' o 'sgd'.")

    torch.manual_seed(SEED)
    model.to(DEVICE)
    step_maker = optimizers[optimizer](model.parameters(), lr=lr)
    loss_function = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        start = time.perf_counter()
        model.train()                                  # activa dropout i batch norm
        total_loss, hits, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            step_maker.zero_grad()                     # esborra el repartiment anterior
            scores = model(images)                     # 1. la xarxa prediu
            loss = loss_function(scores, labels)       # 2. mesurem l'error (la loss)
            loss.backward()                            # 3. backpropagation: repartim la culpa
            step_maker.step()                          # 4. una passa de gradient descent
            total_loss += loss.item() * len(labels)
            hits += (scores.argmax(dim=1) == labels).sum().item()
            total += len(labels)

        val_loss, val_acc = _evaluate(model, val_loader)
        history["train_loss"].append(total_loss / total)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(hits / total)
        history["val_acc"].append(val_acc)
        print(f"Epoch {epoch}/{epochs} · loss {total_loss / total:.3f} · "
              f"encert a validació {val_acc:.1%} · {time.perf_counter() - start:.0f} s")

        if not math.isfinite(total_loss):
            print("La loss s'ha disparat i l'entrenament s'atura: "
                  "segurament el learning rate és massa alt.")
            break
    return history


def plot_history(history):
    """Dibuixa les corbes de loss i d'encert d'un entrenament."""
    fig, grafics = plt.subplots(1, 2, figsize=(10, 3.5))
    epochs = range(1, len(history["train_loss"]) + 1)
    grafics[0].plot(epochs, history["train_loss"], "o-", label="entrenament")
    grafics[0].plot(epochs, history["val_loss"], "o-", label="validació")
    grafics[0].set_ylabel("loss")
    grafics[1].plot(epochs, history["train_acc"], "o-", label="entrenament")
    grafics[1].plot(epochs, history["val_acc"], "o-", label="validació")
    grafics[1].set_ylabel("encert")
    for grafic in grafics:
        grafic.set_xlabel("epoch")
        grafic.legend()
    plt.tight_layout()
    plt.show()

I aquestes són unes quantes funcions per mirar dins de la xarxa i moure imatges.
Tampoc cal modificar-les.

In [ ]:
# Funcions per mirar dins de la xarxa. No cal modificar-la.
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


def score(model, X, y):
    """Retorna l'encert del model amb les imatges X i les etiquetes y."""
    return accuracy(model, DataLoader(TensorDataset(X, y), batch_size=1000))


def shift_images(X, pixels):
    """Desplaça les imatges cap a la dreta i omple l'esquerra de negre (píxels a 0)."""
    if pixels == 0:
        return X
    shifted = torch.zeros_like(X)
    shifted[..., pixels:] = X[..., :-pixels]
    return shifted


def show_filters(model):
    """Dibuixa els filtres apresos a la primera capa: vermell, pes positiu; blau, negatiu."""
    weights = model[0].weight.detach().cpu()[:, 0].numpy()
    fig, grafics = plt.subplots(2, len(weights) // 2, figsize=(len(weights) * 0.65, 3))
    for kernel, grafic in zip(weights, grafics.flat):
        limit = abs(kernel).max()
        grafic.imshow(kernel, cmap="RdBu_r", vmin=-limit, vmax=limit)
        grafic.axis("off")
    fig.suptitle("Filtres apresos a la primera capa")
    plt.show()


@torch.no_grad()
def show_feature_maps(model, image, conv=1, n_maps=8):
    """Dibuixa què detecten els filtres d'una capa convolucional en una imatge.

    Paràmetres:
        model: la CNN entrenada.
        image: una imatge, de forma (canals, alt, ample).
        conv: quina capa convolucional es mira (1 és la primera).
        n_maps: quants mapes de característiques es dibuixen.
    """
    positions = [i for i, layer in enumerate(model) if isinstance(layer, nn.Conv2d)]
    if not 1 <= conv <= len(positions):
        raise ValueError(f"Aquest model té {len(positions)} capes convolucionals: tria un conv entre 1 i {len(positions)}.")
    model.eval()
    device = next(model.parameters()).device
    upto = next(i for i, layer in enumerate(model) if i > positions[conv - 1] and isinstance(layer, nn.ReLU)) + 1
    maps = model[:upto](image.unsqueeze(0).to(device))[0].cpu()
    n_maps = min(n_maps, len(maps))
    fig, grafics = plt.subplots(1, n_maps + 1, figsize=(1.5 * (n_maps + 1), 2))
    grafics[0].imshow(image.permute(1, 2, 0).squeeze().cpu().numpy(), cmap="gray")
    grafics[0].set_title("imatge")
    for number, grafic in enumerate(grafics[1:]):
        grafic.imshow(maps[number].numpy(), cmap="magma")
        grafic.set_title(f"filtre {number}", fontsize=9)
    for grafic in grafics:
        grafic.axis("off")
    fig.suptitle(f"Mapes de la capa convolucional {conv}: {maps.shape[1]} × {maps.shape[2]} píxels", y=1.05)
    plt.show()

## 1. Les mateixes dades, amb una forma nova

Fem servir el mateix MNIST del NB 1.2, amb la mateixa partició: 50.000 imatges per
entrenar, 10.000 per validar i el test segellat. Hi ha un sol canvi, i és important.

#### Un incís: què és un tensor

Un **tensor** és un array de NumPy amb dos extres: pot viure a la GPU i recorda els càlculs
per fer la backpropagation. Per a tu és el mateix: una graella de números amb una forma.

**L'MLP rebia cada imatge en fila**, amb forma `(50000, 784)`. **Una CNN la rep amb la
seva geometria**, amb forma `(50000, 1, 28, 28)`: 50.000 imatges, 1 canal, 28 píxels d'alt i
28 d'ample. El **canal** és quantes capes de color té una imatge: 1 en gris i 3 en color,
com veurem amb CIFAR-10 al NB 6.2. Sense aquesta forma, la xarxa no sabria quins píxels
són veïns, que és justament el problema que volem resoldre.

In [ ]:
with urllib.request.urlopen(URL_DATA) as response:
    mnist = np.load(io.BytesIO(response.read()))


def to_images(X):
    """Passa de files de 784 enters a tensors de forma (N, 1, 28, 28) amb valors de 0 a 1."""
    return torch.tensor(X.reshape(-1, 1, 28, 28) / 255.0, dtype=torch.float32)


X_full, y_full = to_images(mnist["X_train"]), torch.tensor(mnist["y_train"].astype(int))
X_train, y_train = X_full[:50_000], y_full[:50_000]
X_val, y_val = X_full[50_000:], y_full[50_000:]

train_set = TensorDataset(X_train, y_train)
val_set = TensorDataset(X_val, y_val)
train_loader, val_loader = make_loaders(train_set, val_set)

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)

`TensorDataset` ajunta imatges i etiquetes. `make_loaders` les reparteix en **mini-batches**
de 64 imatges, com feia `batch_size` a `MLPClassifier`, i les barreja a cada epoch.

## 2. La primera CNN

Executar primer, entendre després. Creem la xarxa i n'imprimim les capes.

In [ ]:
model = make_cnn(in_channels=1, n_classes=10)
print(model)
print()
print("Pesos:", count_parameters(model))

Té **105.866 pesos**, molt a prop dels 101.770 de l'MLP del NB 1.2. Així la comparació és
justa: mateixa mida, arquitectura diferent. Ara l'entrenem.

> ⏱ Uns 30 segons a un ordinador normal, i potser un minut i mig al CPU de Colab.

In [ ]:
history = train_model(model, train_loader, val_loader, epochs=3, lr=0.001)
plot_history(history)

**98,5% a validació en només 3 epochs.** L'MLP arribava al 97,6% en 10. Amb gairebé els
mateixos pesos, la CNN passa de fer unes 24 errades per cada 1.000 imatges a fer-ne 15.

Què ha canviat respecte a scikit-learn? Menys del que sembla:

| Què vols fer | Amb `MLPClassifier` | Amb la plantilla |
|---|---|---|
| Triar la mida de la xarxa | `hidden_layer_sizes=(128,)` | `filters=(16, 32)` |
| Crear el model | `MLPClassifier(...)` | `make_cnn(...)` |
| Fixar el batch | `batch_size=64` | `make_loaders(..., batch_size=64)` |
| Entrenar | `fit(X, y)` | `train_model(...)` |
| Fixar el learning rate i les epochs | Paràmetres del constructor | Paràmetres de `train_model` |
| Mesurar l'encert | `score(X, y)` | `accuracy(model, loader)` o `score(model, X, y)` |

#### Un incís: Adam

`train_model` fa servir per defecte **Adam** en lloc del gradient descent del NB 1.2. Adam
és una variant que **adapta la longitud de la passa per a cada pes**: és com baixar la
muntanya a les palpentes fent passes més curtes on el terreny és traïdor. Per això hi va bé
un learning rate de 0,001, cent vegades més petit que el 0,1 que fèiem servir. El
compararem amb SGD al NB 6.2.

### Ara tu · Missió 1: troba el learning rate

Ja sabem del NB 1.1 que el learning rate pot fer que una xarxa no aprengui res. Comprovem si
amb una CNN i Adam passa el mateix.

1. **Aposta.** Dels valors 0,1, 0,01, 0,001 i 0,0001, quin creus que donarà el millor encert
   de validació després de 2 epochs? I el pitjor? Apunta-ho.
2. **Experimenta.** Per a cada valor, crea una xarxa nova amb `make_cnn(1, 10)` i entrena-la
   2 epochs amb `train_model(model, train_loader, val_loader, epochs=2, lr=...)`.
   Compte: cal una xarxa nova cada vegada; si no, continuaries entrenant l'anterior.
3. **Compara.** Fes una taula amb el learning rate i l'últim valor de `history["val_acc"]`.
4. **Explica.** Has encertat? Relaciona el pitjor resultat amb les valls del NB 1.1.

> ⏱ Són quatre entrenaments de 2 epochs: un minut i mig a un ordinador normal.

In [ ]:
# El teu codi aquí

## 3. La lupa

Ara sí: què fa aquesta xarxa? Una neurona d'un MLP mira **tota la imatge alhora, en unes
posicions fixes**. Un **filtre convolucional** és un quadradet de pesos, aquí de 3 × 3, que
**recorre tota la imatge com una lupa**. A cada posició multiplica els seus pesos pels píxels
que té a sota, ho suma i apunta el resultat.

Un exemple amb números. Aquest filtre té negatius a l'esquerra i positius a la dreta:

| | | |
|---|---|---|
| −1 | 0 | 1 |
| −1 | 0 | 1 |
| −1 | 0 | 1 |

| Què hi ha sota la lupa | Càlcul | Resultat |
|---|---|---|
| Tot negre: tres files de `0 0 0` | −0 + 0 + 0, tres vegades | **0** |
| Tot gris: tres files de `5 5 5` | −5 + 0 + 5, tres vegades | **0** |
| Una vora: tres files de `0 0 9` | −0 + 0 + 9, tres vegades | **27** |

Només respon on hi ha **una vora vertical**. El resultat de recórrer tota la imatge és una
altra imatge, el **mapa de característiques**, que s'encén on ha trobat aquell patró.

**Per què cal?** Perquè resol els dos problemes de l'MLP:

- **Veïns:** el filtre sempre mira píxels que es toquen.
- **Posició:** el mateix filtre passa per tot arreu. Si el traç es mou, el troba més enllà.

I el més important: **ningú no tria aquests pesos**. La xarxa els aprèn amb la
backpropagation, com els de l'MLP. Vegem els 16 filtres que ha après la primera capa.

In [ ]:
show_filters(model)

Vermell vol dir pes positiu i blau, negatiu. Allà on passa de blau a vermell, el filtre
respon a una vora, com el del nostre exemple, però amb altres orientacions. Ara mirem què veu cada filtre en una imatge.

In [ ]:
show_feature_maps(model, X_val[0], conv=1)
print("Dígit real:", y_val[0].item())

Cada mapa és el mateix dígit vist per un filtre diferent: uns ressalten un costat del traç,
altres el traç sencer, i n'hi ha que s'encenen justament al fons.

### Ara tu · Missió 2: obre la xarxa

La funció `show_feature_maps(model, image, conv=1, n_maps=8)` et deixa mirar qualsevol capa.

1. Tria dues imatges de validació que semblin diferents, per exemple un 1 i un 8. Per trobar
   la primera imatge d'un dígit: `X_val[y_val == 8][0]`.
2. **Aposta.** A la capa 2, els mapes s'assemblaran més o menys al dígit que a la capa 1?
3. Dibuixa els mapes de les dues capes per a cada imatge.
4. **Explica.** Quina mida tenen els mapes de cada capa? (Surt al títol del gràfic.) Hi ha
   algun filtre que s'encengui molt amb un dígit i gairebé gens amb l'altre? Què creus que
   busca?

In [ ]:
# El teu codi aquí

## 4. Per què calen tan pocs pesos

Mira quants pesos gasta cada part del nostre model.

In [ ]:
for position, layer in enumerate(model):
    n_weights = sum(weights.numel() for weights in layer.parameters())
    if n_weights:
        print(f"Capa {position:2d} {layer.__class__.__name__:8s}: {n_weights:7d} pesos")

Les dues capes convolucionals, que miren tota la imatge, només gasten **4.800 pesos**. La resta,
més de 100.000, se'ls emporten les capes finals, les mateixes que ja coneixes de l'MLP, que
decideixen a partir del que han trobat els filtres.

Això es diu **compartir pesos**. Si un detector de vores servís només per a una posició,
necessitaríem un detector per a cada lloc de la imatge. Com que el mateix filtre passa per
tot arreu, n'hi ha prou amb un. Una sola capa de l'MLP, de 784 a 128 neurones, gasta 100.480
pesos; la primera capa convolucional, 160.

**La idea de fons de la UT5 es manté:** les capes ocultes aprenen les característiques. Aquí
la primera capa aprèn vores, i la següent les combina en corbes i cantonades.

### Ara tu · Missió 3: fes-la créixer

El paràmetre `filters` és com `hidden_layer_sizes`: `(16, 32)` són dues capes convolucionals, de 16 i
32 filtres.

1. **Aposta.** Ordena aquestes quatre configuracions de més a menys encert, i de més a menys
   pesos: `(8, 16)`, `(16, 32)`, `(32, 64)` i `(16, 32, 64)`.
2. **Experimenta.** Per a cadascuna, crea-la amb `make_cnn(1, 10, filters=...)`, entrena-la 2
   epochs i mesura el temps amb `time.perf_counter()`, com al NB 1.2.
3. **Fes una taula** amb filtres, pesos (`count_parameters`), temps i encert de validació.
4. **Explica.** La xarxa amb tres capes té més o menys pesos que la de `(32, 64)`? Per què?
   Pista: mira la mida de la imatge just abans de les capes finals, amb `print(model)` i el
   que has vist a la missió 2. Quina triaries per posar-la al mòbil?

> ⏱ Són quatre entrenaments de 2 epochs: uns dos minuts a un ordinador normal.

In [ ]:
# El teu codi aquí

## 5. El pooling: reduir sense perdre el que importa

Entre capa i capa, la xarxa porta una peça que encara no hem explicat: `MaxPool2d`. El
**pooling** divideix la imatge en quadradets de 2 × 2 i de cadascun només es queda el valor més gran.

| Quadradet del mapa | Què en queda |
|---|---|
| `0 1` / `0 9` | **9** |
| `0 0` / `0 0` | **0** |

**Per a què serveix?** El mapa de 28 × 28 passa a 14 × 14 i després a 7 × 7. Té dues
conseqüències:

- **Menys números** per a les capes finals, i per tant molts menys pesos.
- **Més tolerància.** El 9 es conserva encara que es mogui un píxel dins del quadradet.

A més, com que cada capa veu una imatge més petita, la seva lupa de 3 × 3 cobreix una zona
més gran de l'original: la primera capa veu traços; la segona, formes.

### Ara tu · Missió 4: sense pooling

1. **Aposta.** Si treiem el pooling amb `make_cnn(1, 10, pooling=False)`, la xarxa tindrà més
   o menys pesos? Quants cops? L'encert pujarà o baixarà? I el temps?
2. **Experimenta.** Mira'n els pesos amb `count_parameters` i entrena-la **1 sola epoch**. Fes el
   mateix amb `make_cnn(1, 10)`, i mesura el temps de totes dues.
3. **Compara** pesos, temps i encert de validació en una taula.
4. **Explica.** D'on surten tots aquests pesos de més? Pista: la capa que els gasta és la
   mateixa que has vist a la secció 4. Justificaries el preu d'aquesta xarxa?

> ⏱ Tots dos entrenaments junts tarden uns 40 segons a un ordinador normal.

In [ ]:
# El teu codi aquí

## 6. El moment de la veritat: el dígit desplaçat

Hem arribat a l'experiment que va fer caure l'MLP. Movem totes les imatges de validació i
mesurem. **Abans d'executar, aposta:** amb 4 píxels, quant creus que encertarà?

In [ ]:
rows = []
for pixels in range(5):
    rows.append({"desplaçament (píxels)": pixels, "CNN (16, 32)": score(model, shift_images(X_val, pixels), y_val)})
shift_table = pd.DataFrame(rows).set_index("desplaçament (píxels)").round(3)
shift_table

**Aguanta molt millor que l'MLP.** Allà, amb 3 píxels cap model no arribava al 55%, i amb 4
quedaven entre el 24% i el 36%. Aquí, el 89,6% i el 70,4%.

**Però fixa't que també cau.** Al NB 1.2 vam dir que, si el pal d'un 1 es movia, la lupa el
trobava igualment. És cert: el filtre el troba. **El problema ve després.** Abans de les
capes finals, `Flatten` torna a posar els mapes en una fila, i aquelles capes aprenen de
nou en quina posició esperen trobar cada cosa. La part convolucional no té un problema
amb la posició, però l'última part sí.

Hi ha una alternativa: el **global average pooling**. En lloc d'aplanar els mapes, es fa
**la mitjana de cada mapa**, i queda un sol número per filtre. Aquest número diu «he vist
aquest patró» i no «l'he vist en aquest lloc».

| Sortida d'un filtre en un mapa de 7 × 7 | `Flatten` | Global average pooling |
|---|---|---|
| Quants números en surten | 49 | 1 |
| Sap on era el patró? | Sí | No |

A `make_cnn` s'activa amb `global_pool=True`. Sense capes finals grans, també necessita
molts menys pesos.

### Ara tu · Missió 5: la CNN que no mira on

1. **Aposta.** Amb `make_cnn(1, 10, filters=(16, 32, 64), global_pool=True)`, l'encert sense
   desplaçar serà més alt o més baix que el de `model`? I amb 4 píxels?
2. **Experimenta.** Crea-la, mira'n els pesos i entrena-la 3 epochs.
3. **Omple la taula.** Afegeix una columna a `shift_table` amb l'encert per a cada desplaçament
   de 0 a 4, fent servir `score` i `shift_images`.
4. **Dibuixa** les dues columnes en un gràfic, amb el desplaçament a l'eix horitzontal.
5. **Explica.** Quin dels dos models posaries a llegir dígits que no sempre estan centrats? Què
   hi has guanyat i què hi has perdut? Què hauries de provar abans de decidir? Pista: quantes
   epochs ha entrenat cada model?

> ⏱ Un minut a un ordinador normal.

In [ ]:
# El teu codi aquí

## 7. L'experiment contrari: barrejar els píxels

A l'exercici 4 del NB 1.2 vam barrejar els píxels de totes les imatges amb una mateixa
recepta. Per a nosaltres quedaven il·legibles, però l'MLP no ho notava gaire: els píxels
barrejats sempre queien al mateix lloc. **I una CNN?** La recepta trenca precisament el que
aprofita la lupa: que els píxels del costat siguin veïns de veritat.

Aquesta cel·la prepara les imatges barrejades.

In [ ]:
rng = np.random.default_rng(0)
recipe = rng.permutation(784)     # la mateixa recepta del NB 1.2


def shuffle_pixels(X):
    """Aplica la mateixa recepta de barreja a totes les imatges."""
    return X.flatten(start_dim=1)[:, recipe].reshape(-1, 1, 28, 28)


X_train_shuffled, X_val_shuffled = shuffle_pixels(X_train), shuffle_pixels(X_val)
shuffled_loader, shuffled_val_loader = make_loaders(TensorDataset(X_train_shuffled, y_train),
                                                    TensorDataset(X_val_shuffled, y_val))

fig, grafics = plt.subplots(2, 6, figsize=(9, 3.2))
for col in range(6):
    grafics[0, col].imshow(X_val[col, 0].numpy(), cmap="gray")
    grafics[1, col].imshow(X_val_shuffled[col, 0].numpy(), cmap="gray")
    grafics[0, col].axis("off")
    grafics[1, col].axis("off")
plt.show()

### Ara tu · Missió 6: treu-li els veïns

1. **Aposta.** Si entrenes la CNN amb les imatges barrejades, quin encert tindrà: com l'original,
   com l'MLP del NB 1.2 (97,6%), o molt menys? Apunta un número.
2. **Experimenta.** Crea una xarxa nova amb `make_cnn(1, 10)` i entrena-la 3 epochs amb
   `shuffled_loader` i `shuffled_val_loader`.
3. **Compara** amb la CNN original (secció 2) i amb l'MLP. Què ha perdut la CNN?
4. **Mira-ho per dins.** Dibuixa els mapes de la capa 1 amb `show_feature_maps` sobre
   `X_val_shuffled[0]`. S'hi veu res semblant a una vora?
5. **Explica.** Has encertat l'aposta? En quin tipus de dades creus que una CNN no té cap
   avantatge sobre un MLP? Pensa en la taula dels pingüins.

> ⏱ Uns 30 segons a un ordinador normal.

In [ ]:
# El teu codi aquí

## 8. Resum

| | MLP (NB 1.2) | CNN (aquest notebook) |
|---|---|---|
| **Com veu la imatge** | Una fila de 784 números | Una graella, amb veïns |
| **Què aprèn** | Pesos per a posicions fixes | Filtres que recorren la imatge |
| **Pesos** | 101.770 | 105.866 |
| **Encert de validació** | 97,6% en 10 epochs | 98,5% en 3 epochs |
| **Dígit desplaçat 4 píxels** | Entre el 24% i el 36% | 70,4% |

### Les idees que t'has d'endur

1. **Un filtre és una lupa de pesos** que recorre la imatge. El seu resultat és un mapa de
   característiques.
2. **Compartir pesos** fa que mirar tota la imatge costi molt poc.
3. **El pooling** redueix els mapes: menys pesos, més tolerància i més camp de visió.
4. **La lupa no resol sola la posició.** `Flatten` la torna a fixar. El global average pooling
   hi renuncia, a canvi d'altres coses.
5. **L'avantatge d'una CNN ve d'una suposició:** que els píxels veïns tenen relació. Si no és
   certa, l'avantatge desapareix.

## Exercicis finals

Les sis missions les has fet pel camí. Aquests dos exercicis et demanen tot el notebook.

**Exercici 7. La CNN més petita.** Busca la configuració de `make_cnn` amb **menys pesos** que
superi el 98% de validació en un màxim de 5 epochs. Pots fer servir `filters`, `pooling` i
`global_pool`. Compara el teu resultat amb el de les persones del costat.

**Exercici 8. Obrim el test.** Tria el teu millor model i, només aquest, mesura'l amb el test
segellat: `to_images(mnist["X_test"])` i `torch.tensor(mnist["y_test"].astype(int))`, amb `score`.
Com a la secció 9 del NB 1.2, compara-ho amb la validació. S'hi assembla? Per què només s'ha de
fer amb el model final?

## Per al debat de classe

1. **L'MLP té 100.000 pesos que no s'aprofiten gaire per a imatges.** Quina altra mena de
   dades creus que tenen «veïns»? Hi aplicaries una lupa?
2. **Hem triat el model comparant uns pocs números.** Si un model encerta el 99% però falla
   amb dígits moguts, i un altre encerta el 94% però no se n'immuta, quin és millor? De què depèn?